# Comparación de arquitecturas convolucionales con CIFAR-10

## Objetivo

Construiremos y compararemos **dos redes convolucionales desde cero** y una red que reutiliza conocimiento aprendido previamente (*transfer learning*). Al final mediremos su desempeño en imágenes que no usaron para aprender.

**Pregunta de investigación:** ¿qué arquitectura clasifica mejor CIFAR-10 y qué costo tiene en tiempo y cantidad de parámetros?

> Este cuaderno está pensado para ejecutarse en Google Colab con acelerador GPU: abre **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. Ejecuta las celdas en orden. El entrenamiento tarda varios minutos y los resultados dependen del hardware y de la aleatoriedad.

## ¿Qué es una CNN?

Una red neuronal convolucional aplica pequeños filtros que recorren una imagen. Las primeras capas suelen detectar patrones simples (bordes y colores); capas más profundas combinan esos patrones para reconocer formas y objetos. *Pooling* reduce el tamaño espacial de los mapas de características. Al final, una capa de clasificación asigna probabilidades a las categorías.

**CIFAR-10** contiene 60 000 imágenes a color de 32 × 32 píxeles, repartidas en 10 clases. Usaremos 45 000 imágenes para entrenar, 5 000 para elegir cuándo detenernos y 10 000 de prueba para la evaluación final.

## Plan experimental

1. **CNN pequeña:** dos bloques convolucionales; sirve como línea base.
2. **CNN regularizada:** más bloques y filtros, *Batch Normalization*, *Dropout* y aumento de datos.
3. **MobileNetV2 preentrenada:** congela sus filtros aprendidos con ImageNet y aprende una cabeza nueva para las 10 clases. Como CIFAR-10 es pequeño, agrandamos las imágenes a 96 × 96 antes de pasarlas por la red.

Usaremos la misma división de datos, función de pérdida y métrica de exactitud. *Data augmentation* transforma imágenes solo durante el entrenamiento; la validación y la prueba conservan imágenes originales. **No se debe escoger el mejor modelo mirando el conjunto de prueba**: se reserva para la comparación final.


## 1. Preparación e inspección de datos

`Rescaling` convierte los píxeles de `[0, 255]` a `[0, 1]`. La pérdida `SparseCategoricalCrossentropy` recibe etiquetas como enteros (0 a 9), así que no hace falta convertirlas a vectores *one-hot*. Fijar una semilla ayuda a reproducir la división y la inicialización, aunque no garantiza resultados idénticos en todos los aceleradores.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

# CIFAR-10 viene incluido en Keras; la primera ejecución descarga los datos.
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_all = y_train_all.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")

# Separación fija: 45 000 entrenamiento + 5 000 validación.
x_train, y_train = x_train_all[:45_000], y_train_all[:45_000]
x_val, y_val = x_train_all[45_000:], y_train_all[45_000:]

class_names = ["avión", "automóvil", "ave", "gato", "ciervo",
               "perro", "rana", "caballo", "barco", "camión"]
print(f"Entrenamiento: {x_train.shape}; validación: {x_val.shape}; prueba: {x_test.shape}")

# Muestra una imagen de cada clase para familiarizarnos con los datos.
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for class_id, ax in enumerate(axes.flat):
    idx = np.flatnonzero(y_train == class_id)[0]
    ax.imshow(x_train[idx])
    ax.set_title(class_names[class_id])
    ax.axis("off")
plt.suptitle("Ejemplos de CIFAR-10")
plt.tight_layout()
plt.show()

## 2. Aumento de datos y modelos

Un volteo o un pequeño desplazamiento crea una variante plausible de una imagen de entrenamiento. Esto dificulta que la red memorice ejemplos exactos y puede mejorar la generalización. En TensorFlow, estas capas se comportan de manera distinta durante entrenamiento y evaluación: en evaluación dejan la imagen intacta.

`BatchNormalization` estabiliza las activaciones entre capas. `Dropout` apaga aleatoriamente parte de las unidades durante el entrenamiento. Son formas de regularización; no son sustitutos de datos de validación.

In [ ]:
# Aumento moderado; se aplica dentro del modelo, solo mientras entrena.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.05),
], name="aumento_datos")


def compile_model(model, learning_rate=1e-3):
    """Métrica y pérdida adecuadas para etiquetas enteras de 10 clases."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def build_cnn_baseline():
    """CNN sencilla: referencia para medir qué aportan más capacidad y regularización."""
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return compile_model(keras.Model(inputs, outputs, name="CNN_base"))


def build_cnn_regularized():
    """CNN más profunda; aumento de datos, normalización y dropout reducen sobreajuste."""
    inputs = keras.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.20)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return compile_model(keras.Model(inputs, outputs, name="CNN_regularizada"))


def build_mobilenet_transfer():
    """MobileNetV2 ImageNet congelada + cabeza nueva para las clases de CIFAR-10."""
    inputs = keras.Input(shape=(32, 32, 3))  # valores de entrada en [0, 1]
    x = data_augmentation(inputs)
    x = layers.Resizing(96, 96)(x)
    # MobileNetV2 espera valores en [-1, 1]; nuestros píxeles ya están en [0, 1].
    x = layers.Rescaling(scale=2.0, offset=-1.0)(x)
    backbone = keras.applications.MobileNetV2(
        input_shape=(96, 96, 3), include_top=False, weights="imagenet")
    backbone.trainable = False  # conserva los filtros preentrenados
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="MobileNetV2_transfer")
    return compile_model(model, learning_rate=1e-3)

models = {
    "CNN base": build_cnn_baseline(),
    "CNN regularizada": build_cnn_regularized(),
    "MobileNetV2 transfer": build_mobilenet_transfer(),
}
for name, model in models.items():
    print(f"{name}: {model.count_params():,} parámetros totales; "
          f"{sum(np.prod(w.shape) for w in model.trainable_weights):,} entrenables")

## 3. Entrenamiento

`EarlyStopping` detiene el entrenamiento cuando la pérdida de validación deja de mejorar y recupera los mejores pesos vistos. Así limitamos el sobreajuste y evitamos fijar de antemano un número excesivo de épocas. La misma paciencia y máximo de épocas se aplican a los tres modelos.

Al terminar, guardaremos el historial y el tiempo de entrenamiento para comparar también el costo práctico, no solo la exactitud.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
histories = {}
training_seconds = {}

for name, model in models.items():
    print(f"\n{'=' * 18} Entrenando {name} {'=' * 18}")
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.TerminateOnNaN(),
    ]
    start = time.perf_counter()
    histories[name] = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    training_seconds[name] = time.perf_counter() - start
    print(f"Tiempo: {training_seconds[name] / 60:.1f} min; "
          f"épocas ejecutadas: {len(histories[name].history['loss'])}")

## 4. Curvas de aprendizaje

La exactitud de entrenamiento muestra qué tanto aprendió el modelo de los ejemplos vistos. La exactitud de validación aproxima cómo responde a ejemplos no usados para ajustar los pesos. Una brecha persistente (entrenamiento alta y validación baja) suele indicar sobreajuste. Una pérdida de validación que vuelve a subir también puede ser señal de sobreajuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, history in histories.items():
    axes[0].plot(history.history["accuracy"], label=f"{name} · train")
    axes[0].plot(history.history["val_accuracy"], linestyle="--", label=f"{name} · val")
    axes[1].plot(history.history["loss"], label=f"{name} · train")
    axes[1].plot(history.history["val_loss"], linestyle="--", label=f"{name} · val")
axes[0].set(title="Exactitud por época", xlabel="Época", ylabel="Exactitud", ylim=(0, 1))
axes[1].set(title="Pérdida por época", xlabel="Época", ylabel="Entropía cruzada")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Evaluación final en el conjunto de prueba

Ahora evaluamos cada modelo una sola vez en las 10 000 imágenes reservadas. La exactitud es la proporción de imágenes clasificadas correctamente. La pérdida penaliza también las predicciones equivocadas con mucha confianza. El número de parámetros entrenables ayuda a interpretar cuánto tuvo que aprender cada arquitectura; MobileNet reutiliza muchos parámetros que se mantienen congelados.

In [ ]:
rows = []
for name, model in models.items():
    test_loss, test_accuracy = model.evaluate(x_test, y_test, batch_size=BATCH_SIZE, verbose=0)
    best_epoch = int(np.argmin(histories[name].history["val_loss"]) + 1)
    rows.append({
        "Modelo": name,
        "Exactitud validación": max(histories[name].history["val_accuracy"]),
        "Exactitud prueba": test_accuracy,
        "Pérdida prueba": test_loss,
        "Época con menor pérdida val.": best_epoch,
        "Parámetros totales": model.count_params(),
        "Parámetros entrenables": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        "Minutos entrenamiento": training_seconds[name] / 60,
    })
results = pd.DataFrame(rows).sort_values("Exactitud prueba", ascending=False).reset_index(drop=True)
results.style.format({
    "Exactitud validación": "{:.1%}", "Exactitud prueba": "{:.1%}",
    "Pérdida prueba": "{:.3f}", "Minutos entrenamiento": "{:.1f}",
    "Parámetros totales": "{:,.0f}", "Parámetros entrenables": "{:,.0f}",
})

In [ ]:
ax = results.set_index("Modelo")["Exactitud prueba"].plot(
    kind="bar", figsize=(8, 4), color=["#3973ac", "#55a868", "#c44e52"])
ax.set(title="Comparación final en CIFAR-10", ylabel="Exactitud de prueba", xlabel="")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 6. ¿En qué clases se equivoca el mejor modelo?

La matriz de confusión cruza la clase real (filas) con la predicha (columnas). La diagonal representa aciertos; los valores fuera de ella muestran confusiones. Por ejemplo, CIFAR-10 contiene categorías visualmente parecidas como gato y perro.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_name = results.loc[0, "Modelo"]
best_model = models[best_name]
y_pred = np.argmax(best_model.predict(x_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
print("Mejor según exactitud de prueba:", best_name)
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, xticks_rotation=45,
    cmap="Blues", normalize="true", values_format=".2f")
plt.title(f"Matriz de confusión normalizada · {best_name}")
plt.tight_layout()
plt.show()

## 7. Conclusiones (completar después de ejecutar)

No es responsable declarar un ganador antes de entrenar: las cifras de abajo se completan al ejecutar el cuaderno. Compara principalmente la **exactitud de prueba**, y usa la validación para explicar cómo elegiste el mejor modelo. Considera también el tiempo, la brecha entre entrenamiento y validación, y el número de parámetros. Una mejora pequeña de exactitud puede no justificar un modelo mucho más lento.

**Plantilla para el reporte:**

- La mayor exactitud de prueba fue **[modelo]**, con **[porcentaje]**. La exactitud de validación fue **[porcentaje]**.
- Frente a la CNN base (**[porcentaje]**), la CNN regularizada (**[porcentaje]**) [mejoró/empeoró] en **[diferencia en puntos porcentuales]**. Esto sugiere que [interpretación respaldada por las curvas].
- MobileNetV2 obtuvo **[porcentaje]** y tardó **[minutos]**. Su transferencia de aprendizaje [ayudó/no ayudó] en este experimento; un modelo preentrenado no garantiza ganar, especialmente cuando las imágenes son pequeñas y el dominio (ImageNet) difiere.
- En la matriz de confusión, las clases más confundidas fueron **[clases]**. Esto puede deberse a [similitud visual / baja resolución / pocos ejemplos representativos].
- **Mejoras posibles:** ajustar la tasa de aprendizaje y el aumento de datos con validación; descongelar las últimas capas de MobileNetV2 y hacer *fine-tuning* con una tasa pequeña (por ejemplo `1e-5`); entrenar más tiempo con GPU o probar otra arquitectura. Después de descongelar capas, volver a compilar el modelo. La prueba debe seguir reservada para la evaluación final.

### Ideas clave

- Una convolución aprende patrones locales reutilizables en distintas partes de la imagen.
- *Data augmentation*, *Dropout* y *Batch Normalization* ayudan a controlar el sobreajuste, pero se debe confirmar su efecto con validación.
- Transfer learning empieza desde características ya aprendidas y puede reducir el trabajo necesario, aunque adaptar imágenes pequeñas requiere cuidado.
- La mejor decisión depende conjuntamente del desempeño en datos no vistos y del costo de cómputo.